In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import linprog
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Set aesthetic visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# ==========================================
# 1. LOAD & PREPROCESS DATA (Sections 9-11)
# ==========================================
# Load dataset
df = pd.read_csv("airbnb_data.csv")

# Clean column headers
df.columns = df.columns.str.strip().str.lower()

# Standardize price (strip currency symbols if present)
if df['price'].dtype == object:
    df['price'] = df['price'].astype(str).str.replace('$', '').str.replace(',', '').str.strip()
    df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Filter valid entries
df = df.dropna(subset=['price', 'neighbourhood_group', 'room_type'])
df = df[(df['price'] >= 10) & (df['price'] <= 1000)].copy()

# Add standard proxy variables if working from minimal schema
if 'minimum_nights' not in df.columns:
    np.random.seed(42)
    df['minimum_nights'] = np.random.choice([1, 2, 3, 5, 7], size=len(df), p=[0.4, 0.3, 0.15, 0.1, 0.05])
if 'number_of_reviews' not in df.columns:
    np.random.seed(42)
    df['number_of_reviews'] = np.random.poisson(lam=18, size=len(df))
if 'last_review' in df.columns:
    df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')
else:
    df['last_review'] = pd.date_range(start='2024-01-01', periods=len(df), freq='h')

print("=== DATASET OVERVIEW ===")
print("Dataset Shape:", df.shape)
print("\nFirst 5 Records:\n", df.head())
print("\nLast 5 Records:\n", df.tail())
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())

# ==========================================
# 2. EXPLORATORY DATA ANALYSIS (Section 12)
# ==========================================
# Chart 1: Bar chart - Borough Distribution
plt.figure()
df['neighbourhood_group'].value_counts().plot(kind='bar', color='teal')
plt.title("Distribution of Listings by Neighbourhood Group")
plt.xlabel("Borough")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Chart 2: Bar chart - Average Price by Room Type
plt.figure()
df.groupby('room_type')['price'].mean().plot(kind='bar', color='coral')
plt.title("Mean Nightly Price by Room Type")
plt.xlabel("Room Type")
plt.ylabel("Mean Price ($)")
plt.tight_layout()
plt.show()

# Chart 3: Histogram - Price
plt.figure()
plt.hist(df['price'], bins=40, color='skyblue', edgecolor='black')
plt.title("Histogram of Listing Price")
plt.xlabel("Price ($)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# Chart 4: Histogram - Reviews
plt.figure()
plt.hist(df['number_of_reviews'], bins=40, color='lightgreen', edgecolor='black')
plt.title("Histogram of Review Counts")
plt.xlabel("Reviews")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# Chart 5: Box Plot - Price by Room Type
plt.figure()
sns.boxplot(x='room_type', y='price', data=df)
plt.title("Box Plot: Price Variance across Room Types")
plt.tight_layout()
plt.show()

# Chart 6: Pie Chart - Room Type Share
plt.figure()
df['room_type'].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=140)
plt.title("Market Share by Room Type")
plt.ylabel("")
plt.tight_layout()
plt.show()

# Chart 7: Scatter Plot - Minimum Nights vs Price
plt.figure()
plt.scatter(df['minimum_nights'], df['price'], alpha=0.3, color='purple')
plt.title("Scatter Plot: Minimum Nights vs. Price")
plt.xlabel("Minimum Nights")
plt.ylabel("Price ($)")
plt.tight_layout()
plt.show()

# ==========================================
# 3. DESCRIPTIVE STATS & CORRELATION (Sec 13-15)
# ==========================================
num_cols = ['price', 'minimum_nights', 'number_of_reviews']
desc_table = pd.DataFrame({
    'Mean': df[num_cols].mean(),
    'Median': df[num_cols].median(),
    'Mode': df[num_cols].mode().iloc[0],
    'Min': df[num_cols].min(),
    'Max': df[num_cols].max(),
    'Range': df[num_cols].max() - df[num_cols].min(),
    'Variance': df[num_cols].var(),
    'Std Dev': df[num_cols].std()
})
print("\n=== DESCRIPTIVE STATISTICS ===")
print(desc_table)

corr_matrix = df[num_cols].corr()
print("\n=== CORRELATION MATRIX ===")
print(corr_matrix)

plt.figure()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".3f")
plt.title("Correlation Matrix Heatmap")
plt.tight_layout()
plt.show()

# ==========================================
# 4. REGRESSION ANALYSIS (Section 16)
# ==========================================
X = df[['minimum_nights']]
y = df['price']

reg = LinearRegression()
reg.fit(X, y)
y_pred = reg.predict(X)

print("\n=== LINEAR REGRESSION ===")
print(f"Regression Equation: Price = {reg.intercept_:.2f} + ({reg.coef_[0]:.2f} * Minimum_Nights)")
print(f"R² Score: {r2_score(y, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y, y_pred)):.2f}")

plt.figure()
plt.scatter(X, y, color='gray', alpha=0.2, label='Observed Data')
plt.plot(X, y_pred, color='red', linewidth=2, label='Fitted Line')
plt.title("Linear Regression: Price vs Minimum Nights")
plt.xlabel("Minimum Nights")
plt.ylabel("Price ($)")
plt.legend()
plt.tight_layout()
plt.show()

# ==========================================
# 5. PROBABILITY & DECISIONS (Sec 18-19)
# ==========================================
norm_sim = np.random.normal(loc=df['price'].mean(), scale=df['price'].std(), size=1000)
unif_sim = np.random.uniform(low=df['price'].min(), high=df['price'].max(), size=1000)
exp_sim = np.random.exponential(scale=df['price'].mean(), size=1000)

plt.figure()
plt.hist(norm_sim, bins=30, alpha=0.5, label='Normal Sim')
plt.hist(unif_sim, bins=30, alpha=0.5, label='Uniform Sim')
plt.hist(exp_sim, bins=30, alpha=0.5, label='Exponential Sim')
plt.legend()
plt.title("Probability Distribution Simulations")
plt.tight_layout()
plt.show()

payoff_upgrade = (0.6 * 50000) + (0.4 * -10000)
payoff_standard = (0.6 * 20000) + (0.4 * 10000)

print("\n=== DECISION TREE (EMV) ===")
print(f"EMV (Upgrade Unit): ${payoff_upgrade:.2f}")
print(f"EMV (Keep Standard): ${payoff_standard:.2f}")
print(f"Recommended Decision: {'Upgrade Unit' if payoff_upgrade > payoff_standard else 'Keep Standard'}")

# ==========================================
# 6. SAMPLING, CLT, BOOTSTRAP, CIs (Sec 20-22)
# ==========================================
sample_means = [df['price'].sample(50).mean() for _ in range(100)]
print("\n=== SAMPLING & CLT ===")
print(f"Mean of Sample Means: {np.mean(sample_means):.2f}")
print(f"Standard Error: {np.std(sample_means) / np.sqrt(50):.2f}")

ci_90 = stats.t.interval(0.90, len(df['price'])-1, loc=np.mean(df['price']), scale=stats.sem(df['price']))
ci_95 = stats.t.interval(0.95, len(df['price'])-1, loc=np.mean(df['price']), scale=stats.sem(df['price']))
ci_99 = stats.t.interval(0.99, len(df['price'])-1, loc=np.mean(df['price']), scale=stats.sem(df['price']))

print(f"90% CI for Mean Price: {ci_90}")
print(f"95% CI for Mean Price: {ci_95}")
print(f"99% CI for Mean Price: {ci_99}")

boot_medians = [np.median(np.random.choice(df['price'], size=len(df), replace=True)) for _ in range(1000)]
boot_ci_95 = (np.percentile(boot_medians, 2.5), np.percentile(boot_medians, 97.5))
print(f"95% Bootstrap CI for Median Price: {boot_ci_95}")

# ==========================================
# 7. HYPOTHESIS & A/B TESTING (Sec 23-24)
# ==========================================
t_stat, p_val_ttest = stats.ttest_1samp(df['price'], popmean=150)
print("\n=== HYPOTHESIS TESTING (1-Sample t-test) ===")
print(f"H0: mu = 150 | H1: mu != 150")
print(f"t-statistic: {t_stat:.4f}, p-value: {p_val_ttest:.4e}")
print("Decision:", "Reject H0" if p_val_ttest < 0.05 else "Fail to Reject H0")

group_a = df[df['neighbourhood_group'] == 'Manhattan']['number_of_reviews'] > 20
group_b = df[df['neighbourhood_group'] == 'Brooklyn']['number_of_reviews'] > 20

count = [group_a.sum(), group_b.sum()]
nobs = [len(group_a), len(group_b)]
z_stat, p_val_ab = proportions_ztest(count, nobs)

print("\n=== A/B TESTING (Two-proportion z-test) ===")
print(f"Conversion Manhattan: {count[0]/nobs[0]:.3f} | Conversion Brooklyn: {count[1]/nobs[1]:.3f}")
print(f"z-statistic: {z_stat:.4f}, p-value: {p_val_ab:.4e}")
print("Decision:", "Significant Difference" if p_val_ab < 0.05 else "No Significant Difference")

# ==========================================
# 8. TIME SERIES ANALYSIS (Sec 25-26)
# ==========================================
ts_data = df.set_index('last_review').resample('W')['price'].count().dropna()
if len(ts_data) > 10:
    train = ts_data.iloc[:-4]
    test = ts_data.iloc[-4:]

    hw_model = ExponentialSmoothing(train, trend='add', seasonal=None).fit()
    hw_pred = hw_model.forecast(4)

    arima_model = ARIMA(train, order=(1, 1, 1)).fit()
    arima_pred = arima_model.forecast(4)

    hw_rmse = np.sqrt(mean_squared_error(test, hw_pred))
    arima_rmse = np.sqrt(mean_squared_error(test, arima_pred))

    print("\n=== TIME SERIES FORECAST ACCURACY ===")
    print(pd.DataFrame({
        'Model': ['Holt-Winters', 'ARIMA(1,1,1)'],
        'RMSE': [hw_rmse, arima_rmse],
        'MAE': [np.mean(np.abs(test - hw_pred)), np.mean(np.abs(test - arima_pred))]
    }))

# ==========================================
# 9. OPTIMIZATION & SIMULATION (Sec 27-28)
# ==========================================
# Maximize 250*x1 + 120*x2 using SciPy (no PuLP version issues)
c = [-250, -120]
A = [[4, 2], [80, 30]]
b = [100, 2000]
bounds = [(0, None), (0, None)]

res = linprog(c, A_ub=A, b_ub=b, bounds=bounds, method="highs", integrality=[1, 1])

print("\n=== OPTIMIZATION MODELING (SciPy HiGHS) ===")
print(f"Optimal Premium Units: {int(res.x[0])}")
print(f"Optimal Standard Units: {int(res.x[1])}")
print(f"Maximized Profit: ${-res.fun:.2f}")

# Monte Carlo simulation
np.random.seed(42)
sim_days = np.random.randint(180, 300, size=1000)
sim_rate = np.random.normal(loc=df['price'].mean(), scale=15, size=1000)
sim_revenue = sim_days * sim_rate

print("\n=== MONTE CARLO SIMULATION ===")
print(f"Expected Annual Revenue: ${np.mean(sim_revenue):.2f}")
print(f"5th Percentile (Risk Floor): ${np.percentile(sim_revenue, 5):.2f}")
print(f"95th Percentile (Optimistic Ceiling): ${np.percentile(sim_revenue, 95):.2f}")

# ==========================================
# 10. ANOVA & FACTORIAL EXPERIMENT (Sec 29-31)
# ==========================================
top_boroughs = ['Manhattan', 'Brooklyn', 'Queens']
df_anova = df[df['neighbourhood_group'].isin(top_boroughs)]
groups = [group['price'].values for _, group in df_anova.groupby('neighbourhood_group')]
f_stat, p_val_anova = stats.f_oneway(*groups)

print("\n=== ONE-WAY ANOVA ===")
print(f"F-Statistic: {f_stat:.4f}, p-value: {p_val_anova:.4e}")
print("Conclusion:", "Significant price difference between boroughs" if p_val_anova < 0.05 else "No significant difference")

df_fact = df[df['neighbourhood_group'].isin(['Manhattan', 'Brooklyn']) & 
             df['room_type'].isin(['Entire home/apt', 'Private room'])]
model_fact = ols('price ~ C(neighbourhood_group) * C(room_type)', data=df_fact).fit()
anova_fact_table = sm.stats.anova_lm(model_fact, typ=2)

print("\n=== FACTORIAL 2x2 EXPERIMENT ===")
print(anova_fact_table)

Dataset Shape: (48645, 16)

First 5 Records:
      id                                              name  host_id  \
0  2539                Clean & quiet apt home by the park     2787   
1  2595                             Skylit Midtown Castle     2845   
2  3647               THE VILLAGE OF HARLEM....NEW YORK !     4632   
3  3831                   Cozy Entire Floor of Brownstone     4869   
4  5022  Entire Apt: Spacious Studio/Loft by central park     7192   

     host_name neighbourhood_group neighbourhood  latitude  longitude  \
0         John            Brooklyn    Kensington  40.64749  -73.97237   
1     Jennifer           Manhattan       Midtown  40.75362  -73.98377   
2    Elisabeth           Manhattan        Harlem  40.80902  -73.94190   
3  LisaRoxanne            Brooklyn  Clinton Hill  40.68514  -73.95976   
4        Laura           Manhattan   East Harlem  40.79851  -73.94399   

         room_type  price  minimum_nights  number_of_reviews last_review  \
0     Private room

C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)



--- Time Series Forecast Accuracy ---
          Model         RMSE          MAE
0  Holt-Winters  2196.434238  2160.994905
1  ARIMA(1,1,1)  2264.499405  2250.239622


TypeError: LpVariable.__init__() takes 2 positional arguments but 5 were given